In [17]:
from music21 import converter, note, chord, key, articulations

# 定义升号和降号的顺序
ORDER_SHARPS = ['F', 'C', 'G', 'D', 'A', 'E', 'B']
ORDER_FLATS = ['B', 'E', 'A', 'D', 'G', 'C', 'F']

# 加载mxl文件
score = converter.parse('example.mxl')  # 替换为您的文件路径

# 分析调号
key_sig = score.analyze('key')
key_str = f"{key_sig.tonic.name} {key_sig.mode}"  # 例如 "A major"

# 根据调号中的升降符号数量确定哪些音符需要升降
sharps = set()
flats = set()

if key_sig.sharps > 0:
    sharps = set(ORDER_SHARPS[:key_sig.sharps])
elif key_sig.sharps < 0:
    flats = set(ORDER_FLATS[:abs(key_sig.sharps)])

# 初始化左右手音符列表
right_hand_notes = []
left_hand_notes = []

# 初始化数据列表用于后续的训练数据构造
data = []

# 改进后的函数：根据音高范围判断是右手还是左手
def get_hand(part):
    high_notes = 0
    low_notes = 0
    for n in part.flat.notes:
        if isinstance(n, note.Note):
            if n.pitch.midi >= 60:  # C4及以上
                high_notes += 1
            else:
                low_notes += 1
    if high_notes >= low_notes:
        hand = 'right'
    else:
        hand = 'left'
    print(f"Assigning part '{part.partName}' (ID: {part.id}) to {hand} hand based on pitch range.")
    return hand

# 遍历每个 part
for part_index, part in enumerate(score.parts):
    hand = get_hand(part)

    previous_note = None  # 存储前一个音符，用于处理连线

    for element in part.flat.notes:
        # 处理单个音符
        if isinstance(element, note.Note):
            base_name = element.pitch.name  # 例如 'C', 'F#'
            octave = element.octave

            # 根据调号自动添加升降符号
            if base_name in sharps:
                note_name = f"{base_name}#{octave}"
            elif base_name in flats:
                note_name = f"{base_name}b{octave}"
            else:
                note_name = f"{base_name}{octave}"

            # 如果音符有显式的升降符号，覆盖调号的标记
            if element.pitch.accidental:
                if element.pitch.accidental.name == 'sharp':
                    note_name = f"{base_name}#{octave}"
                elif element.pitch.accidental.name == 'flat':
                    note_name = f"{base_name}b{octave}"
                elif element.pitch.accidental.name == 'natural':
                    note_name = f"{base_name}{octave}"

            # 获取指法信息
            fingering = None
            for articulation in element.articulations:
                if isinstance(articulation, articulations.Fingering):
                    fingering = articulation.fingerNumber  # 获取指法号码
                    break  # 假设每个音符只有一个指法

            # 构建音符字符串并添加调号标记
            note_str = f"{key_str} {note_name}"
            if fingering:
                note_str += f" fingering={fingering}"

            # 处理连线，仅在 'continue' 和 'stop' 类型时添加 "finger stays"
            if element.tie:
                if element.tie.type in ['continue', 'stop']:
                    note_str += " (finger stays)"

            # 检查是否为同音连线的不需要弹奏的音符
            if (previous_note and
                previous_note.name == element.name and
                element.tie and
                element.tie.type == 'start'):
                note_str += " (do not play)"
            else:
                # 根据 hand 分配到左右手
                if hand == 'right':
                    right_hand_notes.append(note_str)
                elif hand == 'left':
                    left_hand_notes.append(note_str)

            # 将数据添加到训练数据列表中
            data.append({
                'note': note_name,
                'octave': octave,
                'duration': element.quarterLength,
                'hand': hand,
                'fingering': fingering
            })

            # 更新前一个音符
            previous_note = element

        # 处理和弦
        elif isinstance(element, chord.Chord):
            chord_notes = []
            do_not_play = False  # 标记是否有音符不需要弹奏

            for n in element.notes:
                base_name = n.pitch.name
                octave = n.octave

                # 根据调号自动添加升降符号
                if base_name in sharps:
                    note_name = f"{base_name}#{octave}"
                elif base_name in flats:
                    note_name = f"{base_name}b{octave}"
                else:
                    note_name = f"{base_name}{octave}"

                # 如果音符有显式的升降符号，覆盖调号的标记
                if n.pitch.accidental:
                    if n.pitch.accidental.name == 'sharp':
                        note_name = f"{base_name}#{octave}"
                    elif n.pitch.accidental.name == 'flat':
                        note_name = f"{base_name}b{octave}"
                    elif n.pitch.accidental.name == 'natural':
                        note_name = f"{base_name}{octave}"

                # 获取指法信息
                fingering = None
                for articulation in n.articulations:
                    if isinstance(articulation, articulations.Fingering):
                        fingering = articulation.fingerNumber  # 获取指法号码
                        break  # 假设每个音符只有一个指法

                # 构建音符字符串并添加调号标记
                note_str = f"{key_str} {note_name}"
                if fingering:
                    note_str += f" fingering={fingering}"

                # 处理连线，仅在 'continue' 和 'stop' 类型时添加 "finger stays"
                if n.tie:
                    if n.tie.type in ['continue', 'stop']:
                        note_str += " (finger stays)"

                # 检查是否为同音连线的不需要弹奏的音符
                if (previous_note and
                    previous_note.name == n.name and
                    n.tie and
                    n.tie.type == 'start'):
                    note_str += " (do not play)"
                    do_not_play = True

                chord_notes.append(note_str)

                # 将数据添加到训练数据列表中
                data.append({
                    'note': note_name,
                    'octave': octave,
                    'duration': n.quarterLength,
                    'hand': hand,
                    'fingering': fingering
                })

            # 构建和弦字符串并添加调号标记
            chord_str = f"{key_str} " + " ".join(chord_notes)

            if do_not_play:
                chord_str += " (do not play)"

            # 根据 hand 分配到左右手
            if hand == 'right':
                right_hand_notes.append(chord_str)
            elif hand == 'left':
                left_hand_notes.append(chord_str)

            # 更新前一个音符为和弦中的最后一个音符
            if element.notes:
                previous_note = element.notes[-1]
            else:
                previous_note = None

        # 处理休止符
        elif isinstance(element, note.Rest):
            rest_duration = element.quarterLength
            # 判断是否是常见的休止符时值
            if rest_duration in [0.25, 0.5, 1, 2, 4, 8]:
                rest_str = f'rest {rest_duration}'
                # 添加到左右手
                right_hand_notes.append(rest_str)
                left_hand_notes.append(rest_str)

# 输出左右手的音符、和弦和休止符
print("右手音符、和弦和休止符：")
for item in right_hand_notes:
    print(item)

print("\n左手音符、和弦和休止符：")
for item in left_hand_notes:
    print(item)

# 查看部分训练数据
print("\n部分训练数据样例：")
for entry in data[:10]:
    print(entry)


Assigning part 'Piano' (ID: P1-Staff1) to right hand based on pitch range.
Assigning part 'Piano' (ID: P1-Staff2) to left hand based on pitch range.
右手音符、和弦和休止符：
E- major B-b4 fingering=2
E- major G5 fingering=5
E- major G5 (finger stays)
E- major F5 fingering=3
E- major G5
E- major F5
E- major E-b5
E- major B-b4
E- major G5
E- major C5 fingering=1
E- major C6
E- major G5
E- major B-b5
E- major A-b5
E- major G5
E- major F5 fingering=1
E- major G5
E- major D5
E- major E-b5
E- major C5 fingering=2
E- major B-b4
E- major D6
E- major C6
E- major B-b5
E- major A-b5
E- major G5
E- major A-b5 fingering=4
E- major C5
E- major D5
E- major E-b5
E- major B-b4
E- major G5
E- major F5
E- major G5
E- major F5 fingering=3
E- major E5
E- major F5
E- major G5
E- major F5
E- major E-b5
E- major E-b5 (finger stays)
E- major F5
E- major E-b5
E- major D5
E- major E-b5 fingering=3
E- major F5
E- major G5
E- major B4
E- major C5
E- major D-b5 fingering=3
E- major C5
E- major F5 fingering=3
E- major E5
E- maj

In [18]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
import numpy as np
import pickle

# 将数据转换为DataFrame
df = pd.DataFrame(data)

# 删除指法缺失的音符
df = df.dropna(subset=['fingering'])

# 确保指法为整数类型
df['fingering'] = df['fingering'].astype(int)

# 初始化LabelEncoder
le_note = LabelEncoder()
le_duration = LabelEncoder()
le_hand = LabelEncoder()
le_fingering = LabelEncoder()

# 对类别特征进行标签编码
df['note_encoded'] = le_note.fit_transform(df['note'])
df['duration_encoded'] = le_duration.fit_transform(df['duration'].astype(str))
df['hand_encoded'] = le_hand.fit_transform(df['hand'])

# 对目标标签进行标签编码
df['fingering_encoded'] = le_fingering.fit_transform(df['fingering'])

# 特征和标签
X = df[['note_encoded', 'duration_encoded', 'hand_encoded']].values
y = df['fingering_encoded'].values

print(f"特征形状: {X.shape}")
print(f"标签形状: {y.shape}")

# 创建序列
sequence_length = 10  # 使用前10个音符预测第11个音符

def create_sequences(X, y, seq_length):
    X_seq = []
    y_seq = []
    for i in range(len(X) - seq_length):
        X_seq.append(X[i:i+seq_length])
        y_seq.append(y[i+seq_length])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X, y, sequence_length)

print(f"序列特征形状: {X_seq.shape}")  # (样本数, sequence_length, 特征数量)
print(f"序列标签形状: {y_seq.shape}")  # (样本数,)

# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(
    X_seq, y_seq, test_size=0.2, random_state=42
)

print(f"训练集样本数: {X_train.shape[0]}")
print(f"验证集样本数: {X_val.shape[0]}")

# 保存LabelEncoder
with open('le_note.pkl', 'wb') as f:
    pickle.dump(le_note, f)

with open('le_duration.pkl', 'wb') as f:
    pickle.dump(le_duration, f)

with open('le_hand.pkl', 'wb') as f:
    pickle.dump(le_hand, f)

with open('le_fingering.pkl', 'wb') as f:
    pickle.dump(le_fingering, f)

# 如果需要，将编码后的数据保存为CSV
# df.to_csv('encoded_fingering_data.csv', index=False)


特征形状: (77, 3)
标签形状: (77,)
序列特征形状: (67, 10, 3)
序列标签形状: (67,)
训练集样本数: 53
验证集样本数: 14
